# Logistic Regression

### Derivation of the Logistic Regression Cost Function

For a feature vector $x$, logistic regression predicts the probability that the output belongs to class $1$.  
The hypothesis function is defined using the **sigmoid function**:

$$
f_w(x) = P(y=1 \mid x; w) \qquad f_w(x) = \sigma(w^T x)
$$

where the sigmoid function is

$$
\sigma(z) = \frac{1}{1 + e^{-z}} \qquad f_w(x) = \frac{1}{1 + e^{-w^T x}}
$$

Since the output variable $y$ is binary, we model it using a **Bernoulli distribution**.  
The probability of observing a label $y$ given input $x$ is below and this can be written in a compact mathematical form as

$$
P(y \mid x;\theta) =
\begin{cases}
h_\theta(x), & \text{if } y=1 \\
1-h_\theta(x), & \text{if } y=0
\end{cases} \qquad P(y \mid x;\theta) = (h_\theta(x))^y (1-h_\theta(x))^{1-y}
$$

Assume we have a dataset with $n$ independent samples: The **likelihood function** is the probability of observing the entire dataset:

$$
L(\theta) = \prod_{i=1}^{n} P(y^{(i)} \mid x^{(i)};\theta) \qquad L(\theta) =
\prod_{i=1}^{n}
(h_\theta(x^{(i)}))^{y^{(i)}}
(1-h_\theta(x^{(i)}))^{1-y^{(i)}}
$$

Maximizing a product is difficult, so we take the logarithm of the likelihood.

$$
\ell(\theta) =
\sum_{i=1}^{n}
\left[
y^{(i)}\log(h_\theta(x^{(i)}))
+
(1-y^{(i)})\log(1-h_\theta(x^{(i)}))
\right]
$$


Logistic regression estimates the parameters by **maximizing the log-likelihood**:

$$
\theta^* = \arg\max_\theta \ell(\theta)
$$

However, machine learning algorithms usually minimize a cost function, so we convert this into a minimization problem.  
The cost function is defined as the **negative log-likelihood**:

$$
J(\theta) = -\ell(\theta) \qquad J(\theta) =
-
\sum_{i=1}^{n}
\left[
y^{(i)}\log(h_\theta(x^{(i)}))
+
(1-y^{(i)})\log(1-h_\theta(x^{(i)}))
\right]
$$

Dividing by the number of samples $n$ gives the average cost:

$$
J(\theta) =
-\frac{1}{n}
\sum_{i=1}^{n}
\left[
y^{(i)}\log(h_\theta(x^{(i)}))
+
(1-y^{(i)})\log(1-h_\theta(x^{(i)}))
\right]
$$

This cost function is commonly known as:

- Log Loss
- Binary Cross Entropy
- Negative Log-Likelihood

### Breast Cancer Prediction (Binary Classification)

In [73]:
# importing essential libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

In [74]:
df = pd.read_csv(r"D:\git things\Machine-Vision-Projects\Datasets\breast-cancer\breast-cancer.csv")
df.head(5)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [75]:
# Changing diagnosis B & M to values of 0 & 1
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})
df["diagnosis"].value_counts()

diagnosis
0    357
1    212
Name: count, dtype: int64

In [76]:
# Selecting features that have correlation over 0.3 with the target variable
corr = df.corr()
corr_target = corr["diagnosis"].abs().sort_values(ascending=False)
print(corr_target.head(25))
y = df["diagnosis"]
selected_features = corr_target[corr_target >= 0.3].index
selected_features = selected_features.drop("diagnosis")
X = df[selected_features]

diagnosis                  1.000000
concave points_worst       0.793566
perimeter_worst            0.782914
concave points_mean        0.776614
radius_worst               0.776454
perimeter_mean             0.742636
area_worst                 0.733825
radius_mean                0.730029
area_mean                  0.708984
concavity_mean             0.696360
concavity_worst            0.659610
compactness_mean           0.596534
compactness_worst          0.590998
radius_se                  0.567134
perimeter_se               0.556141
area_se                    0.548236
texture_worst              0.456903
smoothness_worst           0.421465
symmetry_worst             0.416294
texture_mean               0.415185
concave points_se          0.408042
smoothness_mean            0.358560
symmetry_mean              0.330499
fractal_dimension_worst    0.323872
compactness_se             0.292999
Name: diagnosis, dtype: float64


In [77]:
# Removing highly correlated features (Droping one feature from every highly correlated features with each other)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
X_ = X.drop(columns=to_drop)
print("Removed:", to_drop)
print("Remaining features:", X_.shape[1])

Removed: ['concave points_mean', 'radius_worst', 'perimeter_mean', 'area_worst', 'radius_mean', 'area_mean', 'concavity_mean', 'perimeter_se', 'area_se', 'texture_mean']
Remaining features: 13


In [78]:
# Selecting top features by importance using random forest feature importance
model = RandomForestClassifier(random_state=42)
model.fit(X_, df["diagnosis"])
importance = pd.Series(model.feature_importances_, index=X_.columns)
importance = importance.sort_values(ascending=False)
top_features = importance.index[:10]
X_final = X_[top_features]
print("Final features:", X_final.columns)

Final features: Index(['perimeter_worst', 'concave points_worst', 'concavity_worst',
       'radius_se', 'compactness_worst', 'compactness_mean', 'texture_worst',
       'concave points_se', 'symmetry_worst', 'smoothness_mean'],
      dtype='object')
